In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_date, when, count
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import pandas as pd
import os

# ==========================================================
# 1. KHỞI TẠO SESSION (Nạp Driver Kafka)
# ==========================================================
# Ép nạp package Kafka ngay từ đầu để tránh lỗi "Failed to find data source"
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1 pyspark-shell'

spark = SparkSession.builder \
    .appName("Amazon-Data-Analysis-Notebook") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Spark Session sẵn sàng với Driver Kafka!")

✅ Spark Session sẵn sàng với Driver Kafka!


In [10]:
# ==========================================================
# 2. ĐỊNH NGHĨA SCHEMA (Khớp 100% với dữ liệu thô)
# ==========================================================
kafka_schema = StructType([
    StructField("Order Date", StringType(), True),
    StructField("Purchase Price Per Unit", DoubleType(), True),
    StructField("Quantity", DoubleType(), True),
    StructField("Shipping Address State", StringType(), True),
    StructField("Title", StringType(), True),
    StructField("ASIN/ISBN (Product Code)", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Survey ResponseID", StringType(), True)
])

In [11]:
# ==========================================================
# 3. ĐỌC DỮ LIỆU TỪ KAFKA
# ==========================================================
df_raw = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "amazon_purchases") \
    .option("startingOffsets", "earliest") \
    .load()

# Chuyển đổi và bung các cột
df_amazon = df_raw.selectExpr("CAST(value AS STRING) as raw_str") \
    .select(from_json(col("raw_str"), kafka_schema).alias("data")) \
    .select("data.*")

In [13]:
# ==========================================================
# 4. PHÂN TÍCH TRƯỚC KHI XỬ LÝ (Dùng Pandas)
# ==========================================================
print("\n🔍 --- 10 DÒNG DỮ LIỆU ĐẦU TIÊN ---")
display(df_amazon.limit(10).toPandas())


🔍 --- 10 DÒNG DỮ LIỆU ĐẦU TIÊN ---


,Order Date,Purchase Price Per Unit,Quantity,Shipping Address State,Title,ASIN/ISBN (Product Code),Category,Survey ResponseID
0,2018-12-04,7.98,1.0,NJ,SanDisk Ultra 16GB Class 10 SDHC UHS-I Memory ...,B0143RTB1E,FLASH_MEMORY,R_01vNIayewjIIKMF
1,2018-12-22,13.99,1.0,NJ,Betron BS10 Earphones Wired Headphones in Ear ...,B01MA1MJ6H,HEADPHONES,R_01vNIayewjIIKMF
2,2018-12-24,8.99,1.0,NJ,"""NaN""",B078JZTFN3,"""NaN""",R_01vNIayewjIIKMF
3,2018-12-25,10.45,1.0,NJ,Perfecto Stainless Steel Shaving Bowl. Durable...,B06XWF9HML,DISHWARE_BOWL,R_01vNIayewjIIKMF
4,2018-12-25,10.00,1.0,NJ,Proraso Shaving Cream for Men,B00837ZOI0,SHAVING_AGENT,R_01vNIayewjIIKMF
5,2019-02-18,10.99,1.0,NJ,Micro USB Cable Android Charger - Syncwire [2-...,B01GFB2E9M,COMPUTER_PROCESSOR,R_01vNIayewjIIKMF
6,2019-02-18,4.99,1.0,NJ,Amazon Basics USB 2.0 Charger Cable - A-Male t...,B00NH13S44,COMPUTER_ADD_ON,R_01vNIayewjIIKMF
7,2019-03-15,124.99,1.0,NJ,"Fire HD 8 Tablet (8"" HD Display, 32 GB, withou...",B077H6L7T9,AMAZON_TABLET,R_01vNIayewjIIKMF
8,2019-04-23,12.99,1.0,NJ,"Men's Leather Belt, Ratchet Dress Belt with Au...",B07L84ZZXC,APPAREL_BELT,R_01vNIayewjIIKMF
9,2019-04-23,24.69,1.0,NJ,"""NaN""",B06XKNWJN2,"""NaN""",R_01vNIayewjIIKMF


In [14]:
# Thống kê kích thước
num_rows = df_amazon.count()
num_cols = len(df_amazon.columns)
print(f"\n📏 Kích thước dữ liệu ban đầu: ({num_rows}, {num_cols})")


📏 Kích thước dữ liệu ban đầu: (1850717, 8)


In [18]:
from pyspark.sql import functions as F

# Đếm Null, NaN kiểu số, và "NaN" (chuỗi có ngoặc kép hoặc không)
null_counts = (
    df_amazon.select([
        F.count(
            F.when(
                F.col(c).isNull() | 
                F.isnan(F.col(c)) | 
                (F.col(c).cast("string") == "NaN") | 
                (F.col(c).cast("string") == '"NaN"'), # Thêm kiểm tra chuỗi có dấu ngoặc kép
                c
            )
        ).alias(c)
        for c in df_amazon.columns
    ])
    .toPandas()
    .melt(var_name="column", value_name="null_or_nan_count")
    .sort_values(by="null_or_nan_count", ascending=False)
)

print("\n📊 THỐNG KÊ GIÁ TRỊ THIẾU/LỖI (ĐÃ FIX NGOẶC KÉP):")
display(null_counts)


📊 THỐNG KÊ GIÁ TRỊ THIẾU/LỖI (ĐÃ FIX NGOẶC KÉP):


,column,null_or_nan_count
4,Title,89740
6,Category,89458
3,Shipping Address State,87812
5,ASIN/ISBN (Product Code),973
0,Order Date,0
1,Purchase Price Per Unit,0
2,Quantity,0
7,Survey ResponseID,0


In [19]:
from pyspark.sql import functions as F

# A. Chuyển kiểu dữ liệu cột ngày
df_amazon = df_amazon.withColumn("Order Date", F.to_date(F.col("Order Date"), "yyyy-MM-dd"))

# B. Ép tất cả các kiểu "NaN" (có ngoặc, không ngoặc, chữ thường) về Null
# Chúng ta đưa tất cả vào một danh sách để replace một lần cho sạch
bad_values = ['"NaN"', 'NaN', 'nan', '"nan"']
df_amazon_pre_clean = df_amazon.replace(bad_values, None)

# C. Loại bỏ toàn bộ dòng có bất kỳ giá trị null nào
df_amazon_clean = df_amazon_pre_clean.na.drop()

# D. Kiểm tra nhanh kết quả
print(f"🚀 Xử lý xong! Số dòng còn lại: {df_amazon_clean.count()}")

🚀 Xử lý xong! Số dòng còn lại: 1675015


In [21]:
print("\n✨ 5 DÒNG DỮ LIỆU SẠCH CUỐI CÙNG:")
display(df_amazon_clean.limit(5).toPandas())


✨ 5 DÒNG DỮ LIỆU SẠCH CUỐI CÙNG:


,Order Date,Purchase Price Per Unit,Quantity,Shipping Address State,Title,ASIN/ISBN (Product Code),Category,Survey ResponseID
0,2018-12-04,7.98,1.0,NJ,SanDisk Ultra 16GB Class 10 SDHC UHS-I Memory ...,B0143RTB1E,FLASH_MEMORY,R_01vNIayewjIIKMF
1,2018-12-22,13.99,1.0,NJ,Betron BS10 Earphones Wired Headphones in Ear ...,B01MA1MJ6H,HEADPHONES,R_01vNIayewjIIKMF
2,2018-12-25,10.45,1.0,NJ,Perfecto Stainless Steel Shaving Bowl. Durable...,B06XWF9HML,DISHWARE_BOWL,R_01vNIayewjIIKMF
3,2018-12-25,10.00,1.0,NJ,Proraso Shaving Cream for Men,B00837ZOI0,SHAVING_AGENT,R_01vNIayewjIIKMF
4,2019-02-18,10.99,1.0,NJ,Micro USB Cable Android Charger - Syncwire [2-...,B01GFB2E9M,COMPUTER_PROCESSOR,R_01vNIayewjIIKMF
